# Lecture 2a: Simulating an i.i.d. Data Matrix
**Simulations in Statistics (52001)**

---

## Overview

We construct a function `Part2a` that returns an $n \times d$ **data matrix** of i.i.d. draws from a user-specified distribution.

- **Rows** are independent observations.
- **Columns** are independent and identically distributed variables.

**Arguments:**

- `n` -- integer, number of independent rows (observations).
- `d` -- integer, number of i.i.d. variables stored in each row.
- `rfun` -- character string, name of the R sampler (e.g. `"rnorm"`, `"runif"`, `"rlnorm"`).
- `...` -- additional arguments forwarded to `rfun` (e.g. `meanlog`, `sdlog`).

**Mechanism:** R's `get(rfun)` looks up the function by name in the search list. We draw $n \cdot d$ i.i.d. values in a single call to that sampler and reshape them into an $n \times d$ matrix.

$$X \in \mathbb{R}^{n \times d}, \qquad X_{ij} \stackrel{\text{iid}}{\sim} F_{\text{rfun}}(\,\cdot\,\mid \ldots)$$

**Application:** simulate from the Log-normal distribution with $\mu_{\log} = 3.37$, $\sigma_{\log} = 9.5$ (i.e. $\log X \sim \mathcal{N}(3.37,\ 9.5^2)$), with $n = 10^6$ rows and $d = 5$ columns, then report:

$$\text{(a)}\;\; \operatorname{sd}(X_{\cdot,1}) \qquad \text{(b)}\;\; \operatorname{sd}(\overline{X}_{\text{row}})$$

For Log-normal$(\mu_{\log}, \sigma_{\log})$:
$$\operatorname{Var}(X) = (e^{\sigma_{\log}^2} - 1)\, e^{2\mu_{\log} + \sigma_{\log}^2}$$

With $\sigma_{\log} = 9.5$ this variance is astronomical ($\sigma_X \approx 4.6 \times 10^{40}$). Sample estimates from $n = 10^6$ are dominated by the largest order statistic and will be wildly seed-dependent.

## R Implementation

In [1]:
Part2a <- function(n, d, rfun, ...) {
  sampler      <- get(rfun)
  flat_samples <- sampler(n * d, ...)
  data_matrix  <- matrix(flat_samples, nrow = n, ncol = d)
  return(data_matrix)
}

In [2]:
set.seed(1)

n_rows            <- 10^6
d_cols            <- 5
rfun_name         <- "rlnorm"
lognormal_meanlog <- 3.37
lognormal_sdlog   <- 9.5

X <- Part2a(n = n_rows, d = d_cols, rfun = rfun_name, lognormal_meanlog, lognormal_sdlog)

cat("n            =", n_rows,            "\n")
cat("d            =", d_cols,            "\n")
cat("rfun         =", rfun_name,         "\n")
cat("meanlog (mu) =", lognormal_meanlog, "\n")
cat("sdlog (sig)  =", lognormal_sdlog,   "\n")
cat("dim(X)       =", dim(X),            "\n")

n            = 1e+06 
d            = 5 
rfun         = rlnorm 
meanlog (mu) = 3.37 
sdlog (sig)  = 9.5 
dim(X)       = 1000000 5 


## Computing the Results

In [3]:
sd_first_column <- sd(X[, 1])
sd_row_means    <- sd(rowMeans(X))

cat("a. sd(X[,1])         =", sd_first_column, "\n")
cat("b. sd(rowMeans(X))   =", sd_row_means,    "\n")

a. sd(X[,1])         = 5.336931e+17 
b. sd(rowMeans(X))   = 8.526532e+19 


## Interpretation

- **`sd(X[,1])`** estimates $\sigma_X = \sqrt{(e^{\sigma_{\log}^2} - 1)\, e^{2\mu_{\log} + \sigma_{\log}^2}}$. With $\sigma_{\log} = 9.5$ the theoretical sd is on the order of $10^{40}$, but the sample sd from only $n = 10^6$ values reaches $\sim 10^{17}$ -- determined almost entirely by the single largest draw observed.
- **`sd(rowMeans(X))`** does **not** shrink by $1/\sqrt{d}$ here. The row mean $\overline{X}_{\text{row}}$ is dominated by the largest entry in the row, and pooling $d = 5$ columns gives 5 chances per row to pick up an extreme tail value, so $\operatorname{sd}(\overline{X}_{\text{row}})$ can actually exceed $\operatorname{sd}(X_{\cdot,1})$. The CLT $\sqrt{d}$-rule assumes finite-variance behavior is already realized in the sample, which it is not at this scale.
- The combination of `get(rfun)` and `...` lets `Part2a` work with any sampler whose first argument is the sample size (`rnorm`, `runif`, `rlnorm`, `rexp`, `rbinom`, ...).

## Final Answer

Build $X = $ `Part2a(n = 10^6, d = 5, rfun = "rlnorm", 3.37, 9.5)`.

With `set.seed(1)`:

$$\text{a.}\;\; \operatorname{sd}(X[,1]) \approx 5.34 \times 10^{17} \qquad \text{b.}\;\; \operatorname{sd}(\operatorname{rowMeans}(X)) \approx 8.53 \times 10^{19}$$

Note: because $\sigma_{\log} = 9.5$ produces an extremely heavy-tailed distribution (theoretical $\operatorname{sd}(X) \approx 4.56 \times 10^{40}$), these sample estimates are strongly seed-dependent and reflect the largest order statistics rather than the population sd.